# 03 — Manifest → priors → scale-conditioned mass regressor (GPU job 2)

The novel model: mass regression conditioned on measured metric geometry (docs/MODELS.md §4). Self-contained — stages its own copy of Nutrition5k.

First it **stages Nutrition5k to local disk**, then runs the three pipeline steps: (1) extract the geometry manifest from the RGB-D captures — the same quantities the phone measures; (2) fit the κ/φ/h̄ shape priors used by the pure-geometry pipeline; (3) train the regressor.

- **Storage — read this:** the raw dataset lives on **local disk**, not Drive. Both the manifest extraction and *every training epoch* read the ~5k per-dish RGB-D folders, and that many-small-files workload aborts over Drive's FUSE mount (`Errno 103`, `ECONNABORTED`). Drive is only for the handful of big files worth keeping — the **outputs** (manifest, priors, checkpoint). The manifest bakes in the local `/content/n5k/...` image paths, so a fresh session must rerun the staging cell before re-training.
- **Runtime:** dataset staging minutes–~1 h (bandwidth); manifest extraction ~30–60 min (CPU-bound); training ~1–2 h on H100 at batch 128.
- **Benchmarks to beat:** 26.1% calorie MAPE (RGB) / 16.5% (RGB+depth).

In [32]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/physics-powered-portion-estimation'

# The raw dataset AND framework caches stay on LOCAL disk. Drive's FUSE mount
# aborts (Errno 103, ECONNABORTED) on the per-dish RGB-D workload — thousands
# of tiny files listed and read by both the manifest extraction and every
# training epoch — and mmap over FUSE is unreliable too. Drive is only for the
# few big files worth keeping: the OUTPUTS below. The next cell stages the
# dataset into DATA from the public bucket (reliable HTTPS, resumable).
os.environ['HF_HOME'] = '/content/hf-cache'
os.environ['TORCH_HOME'] = '/content/torch-cache'

DATA = '/content/n5k'                     # local SSD — NOT Drive (see above)
OUT = f'{DRIVE_ROOT}/out'                 # outputs persist to Drive
CKPT = f'{DRIVE_ROOT}/checkpoints/mass-regressor.pt'
!mkdir -p "{OUT}" "{DATA}"
!nvidia-smi | head -12

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Wed Jul  8 21:19:57 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   37C    P0             54W /  400W |       0MiB /  81920MiB |      0%      Default |
|          

In [33]:
# Clone the repo. Private repo → add your GitHub token as a Colab secret named
# GH_TOKEN (🔑 left sidebar). Public repo → no token needed. Fails LOUD.
import os, subprocess
token = ''
try:
    from google.colab import userdata
    token = userdata.get('GH_TOKEN') or ''
except Exception:
    pass
repo = 'github.com/Hakeem-Hannoon/physics-powered-portion-estimation.git'
url = f'https://x-access-token:{token}@{repo}' if token else f'https://{repo}'
# A prior run may leave the kernel's CWD inside /content/ppe (the %cd in the
# next cells). Deleting that dir out from under the kernel makes the git clone
# below fail with "Unable to read current working directory". Reset to a
# stable dir first so re-runs are safe.
os.chdir('/content')
subprocess.run(['rm', '-rf', '/content/ppe'])
r = subprocess.run(['git', 'clone', '--depth', '1', url, '/content/ppe'],
                   capture_output=True, text=True)
assert os.path.isdir('/content/ppe/model'), (
    (r.stderr.replace(token, '***') if token else r.stderr) +
    '\nClone failed — add a GH_TOKEN Colab secret (🔑) or make the repo public.')
print('repo ready')

%pip -q install timm pandas

repo ready


In [44]:
import concurrent.futures, pathlib, requests

BUCKET = 'nutrition5k_dataset'
# Only the slice the pipeline reads — overhead RGB-D, mass/kcal metadata, split
# ids — not the ~160 GB of side-angle video.
PREFIXES = [
    'nutrition5k_dataset/imagery/realsense_overhead/',
    'nutrition5k_dataset/metadata/',
    'nutrition5k_dataset/dish_ids/',
]
API = f'https://storage.googleapis.com/storage/v1/b/{BUCKET}/o'


def list_objects(prefix):
    """Page through the bucket's JSON listing API, yielding (name, size) for
    every object under `prefix`. Unauthenticated GET — the bucket is public."""
    token = None
    while True:
        # maxResults caps the page size; nextPageToken walks to the next page.
        params = {'prefix': prefix, 'fields': 'items(name,size),nextPageToken', 'maxResults': 5000}
        if token:
            params['pageToken'] = token
        r = requests.get(API, params=params, timeout=60)
        r.raise_for_status()
        payload = r.json()
        for item in payload.get('items', []):
            yield item['name'], int(item.get('size', 0))
        token = payload.get('nextPageToken')
        if not token:   # no more pages — done
            break


def fetch(job):
    """Download one object unless an identical-size copy already exists — that
    size check IS the resumability, so a rerun skips completed files. Streams to
    a .part temp and atomically renames, so an interrupted transfer never leaves
    a truncated file a later run would mistake for complete."""
    name, size = job
    dest = pathlib.Path(DATA) / name.removeprefix('nutrition5k_dataset/')
    # Temporarily remove the check for existing files to force a re-download.
    # if dest.exists() and dest.stat().st_size == size:
    #     return 0  # already staged — skip

    dest.parent.mkdir(parents=True, exist_ok=True)
    url = f'https://storage.googleapis.com/{BUCKET}/{name}'
    with requests.get(url, stream=True, timeout=300) as r:
        r.raise_for_status()
        tmp = dest.with_suffix(dest.suffix + '.part')
        with open(tmp, 'wb') as f:
            for chunk in r.iter_content(1 << 20):   # 1 MiB chunks
                f.write(chunk)
        tmp.rename(dest)   # atomic swap — the file appears only once fully written
    return 1 # Return 1 to count as a new file downloaded


# Enumerate everything up front (to print a total), then fetch in parallel —
# 8 threads saturates the bucket without much memory.
jobs = [j for p in PREFIXES for j in list_objects(p)]
print(f'{len(jobs)} objects, {sum(s for _, s in jobs) / 1e9:.1f} GB → {DATA}')
staged = 0
with concurrent.futures.ThreadPoolExecutor(max_workers=8) as pool:
    for i, n in enumerate(pool.map(fetch, jobs), 1):
        staged += n
        if i % 500 == 0:
            print(f'{i}/{len(jobs)} checked, {staged} downloaded')
print(f'staged to local disk: {staged} new files ({len(jobs) - staged} already present)')

10481 objects, 3.2 GB → /content/n5k
500/10481 checked, 500 downloaded
1000/10481 checked, 1000 downloaded
1500/10481 checked, 1500 downloaded
2000/10481 checked, 2000 downloaded
2500/10481 checked, 2500 downloaded
3000/10481 checked, 3000 downloaded
3500/10481 checked, 3500 downloaded
4000/10481 checked, 4000 downloaded
4500/10481 checked, 4500 downloaded
5000/10481 checked, 5000 downloaded
5500/10481 checked, 5500 downloaded
6000/10481 checked, 6000 downloaded
6500/10481 checked, 6500 downloaded
7000/10481 checked, 7000 downloaded
7500/10481 checked, 7500 downloaded
8000/10481 checked, 8000 downloaded
8500/10481 checked, 8500 downloaded
9000/10481 checked, 9000 downloaded
9500/10481 checked, 9500 downloaded
10000/10481 checked, 10000 downloaded
staged to local disk: 10481 new files (0 already present)


In [41]:
import shutil

# Delete the existing local data directory to force a fresh download
if os.path.exists(DATA):
    shutil.rmtree(DATA)
    print(f'Removed existing data directory: {DATA}')
else:
    print(f'Data directory not found, nothing to remove: {DATA}')

# Re-create the directory for the new download
!mkdir -p "{DATA}"

Removed existing data directory: /content/n5k


Now, please re-run the previous cell `F8uCb23ujATz` to re-stage the `Nutrition5k` dataset to local disk, which should resolve the image identification issue and allow the manifest creation to proceed.

In [42]:
# Step 1 — geometry manifest. Runs model/data/prepare_nutrition5k.py: for every
# dish it fits the table plane from the depth map, measures area/height/volume
# (the same physics the phone computes at capture time), and joins the mass/kcal
# labels into one CSV. Reads from local DATA; writes the CSV to OUT on Drive.
# ~30–60 min, CPU-bound. `head -3` is a sanity peek at the columns.
%cd /content/ppe
!python model/data/prepare_nutrition5k.py --root "{DATA}" --out "{OUT}/n5k-manifest.csv"
!head -3 "{OUT}/n5k-manifest.csv"

/content/ppe
Traceback (most recent call last):
  File "/content/ppe/model/data/prepare_nutrition5k.py", line 164, in <module>
    main()
  File "/content/ppe/model/data/prepare_nutrition5k.py", line 120, in main
    for dish_dir in sorted(overhead.iterdir()):
                    ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/pathlib.py", line 1056, in iterdir
    for name in os.listdir(self):
                ^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: '/content/n5k/imagery/realsense_overhead'
head: cannot open '/content/drive/MyDrive/physics-powered-portion-estimation/out/n5k-manifest.csv' for reading: No such file or directory


In [43]:
# Step 2 — shape priors. Runs model/priors/fit_priors.py over the manifest to
# fit the per-class constants κ (V = κ·A^1.5), φ (prism fill fraction), and h̄
# (typical height). The printed global κ replaces DEFAULT_KAPPA in
# packages/pipeline/src/estimate.ts and feeds nutrition/'s shape_priors table.
# Seconds to run; writes priors.json to OUT on Drive.
!python model/priors/fit_priors.py --manifest "{OUT}/n5k-manifest.csv" --out "{OUT}/priors.json"
!cat "{OUT}/priors.json"

Traceback (most recent call last):
  File "/content/ppe/model/priors/fit_priors.py", line 79, in <module>
    main()
  File "/content/ppe/model/priors/fit_priors.py", line 36, in main
    df = pd.read_csv(args.manifest)
         ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pandas/io/parsers/readers.py", line 1026, in read_csv
    return _read(filepath_or_buffer, kwds)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pandas/io/parsers/readers.py", line 620, in _read
    parser = TextFileReader(filepath_or_buffer, **kwds)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pandas/io/parsers/readers.py", line 1620, in __init__
    self._engine = self._make_engine(f, self.engine)
                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pandas/io/parsers/readers.py", line 1880, in _make_engine
    self.handles = get_handle(
 

In [38]:
# Step 3 — train the scale-conditioned mass regressor. Runs
# model/train/mass_regressor_nutrition5k.py: a CNN backbone reads each RGB crop,
# FiLM modulates its features by the measured physics (area/height/scale), and a
# head regresses log(mass) + log(kcal). Per-epoch mass/kcal MAPE prints; the
# best-by-mass-MAPE checkpoint saves to CKPT on Drive. ~1–2 h on an H100.
!python model/train/mass_regressor_nutrition5k.py \
  --manifest "{OUT}/n5k-manifest.csv" \
  --epochs 50 --batch-size 128 \
  --output "{CKPT}"
# Paste the printed best MAPE into the README 'Testing set & results' table:
print('README row template:')
print('| Nutrition5k test split | calorie MAPE | 26.1% RGB / 16.5% depth | **<best kcal MAPE from above>** |')

Traceback (most recent call last):
  File "/content/ppe/model/train/mass_regressor_nutrition5k.py", line 243, in <module>
    main()
  File "/content/ppe/model/train/mass_regressor_nutrition5k.py", line 181, in main
    manifest = pd.read_csv(args.manifest)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pandas/io/parsers/readers.py", line 1026, in read_csv
    return _read(filepath_or_buffer, kwds)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pandas/io/parsers/readers.py", line 620, in _read
    parser = TextFileReader(filepath_or_buffer, **kwds)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pandas/io/parsers/readers.py", line 1620, in __init__
    self._engine = self._make_engine(f, self.engine)
                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pandas/io/parsers/readers.py", line 1880, in _